# 🧠 Brain Tumor MRI — Global Cross-Dataset Duplicate Detector
### Perceptual Hash (pHash) | Priority-Based Deletion | N-Way Leakage Matrix

---
**Strategy:**  
1. Compute `pHash` for every image in all datasets concurrently  
2. Group images by hash — any group with images from **multiple datasets** is a cross-contamination hit  
3. Keep the copy in the **lowest priority number** (highest importance), delete the rest  
4. Report a full **Leakage Matrix** showing which dataset lost how many images to whom

> ⚠️ **Permanent deletion is active.** Duplicates are removed with `os.remove()`. No undo.

## ⚙️ Step 0 — Install Dependencies

In [ ]:
%pip install imagehash 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 10.8 MB/s eta 0:00:00


## 📦 Step 1 — Imports

In [59]:
import os
import sys
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

import imagehash
from PIL import Image
from tqdm.auto import tqdm

print(f"✅ imagehash v{imagehash.__version__} loaded")
print(f"✅ Python {sys.version.split()[0]}")

✅ imagehash v4.3.2 loaded
✅ Python 3.12.13


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("briscdataset/brisc2025")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brisc2025' dataset.
Path to dataset files: /kaggle/input/brisc2025


In [27]:
# Force remove the existing (likely empty or corrupted) folders first
!rm -rf /content/source/train /content/source/test

# Use cp -r (recursive copy) instead of mv
!cp -r /kaggle/input/brisc2025/brisc2025/classification_task/train /content/source/
!cp -r /kaggle/input/brisc2025/brisc2025/classification_task/test /content/source/

# Verify
!ls  /content/source/

test  train


In [4]:
!wget "https://data.mendeley.com/public-api/zip/zwr4ntf94j/download/5" -O dataset.zip


--2026-04-08 12:38:14--  https://data.mendeley.com/public-api/zip/zwr4ntf94j/download/5
Resolving data.mendeley.com (data.mendeley.com)... 162.159.133.86, 162.159.130.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.133.86|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/zwr4ntf94j-5.zip?X-Amz-Security-Token=IQoJb3JpZ2luX2VjEDUaCWV1LXdlc3QtMSJGMEQCIEevIybZ5UN5g4hhsiJ1vhxdRQ2tEGAQgVs5tBSHg2ePAiADfmb3gXmO%2FZUAe8lctUQOLLBEhcgldtAI72J4bTBMgSqVBQj%2B%2F%2F%2F%2F%2F%2F%2F%2F%2F%2F8BEAQaDDM2NzE0NzM4MzgyNSIMl31utVHlVzAFjfiWKukEGP6zndztamfXYb87rtqlM8fy%2BPpKzhq8iWENT2sTUrFmyVtTBankRR6e5awYSIfv19LnRZmz7FpvBqVp4jlvnbyPivHuqSxZwBUWVaS98fAogzk8ATGaiJt9hXMdADyxOrynhDu0cvoBFLljxBWoPcyS8jnPYBk1BERZAFkvkiPSabT9ru4Y%2Ft8dYEHgq%2Bd3D825kZP4eENngyg4%2FSvoABD69rY4cNYx50z%2FwWHgvukBoIErNjptcTQ%2FRYxuGu7%2BJzjuWKFvXhtyhsE6ZvAJmraL0kVBfQGpGU2oRWfOnkocoVbArkCVJBLxuuJZh1jLOYFji8trg%2FAYZhSqpswewHCZ%2

In [5]:
!unzip  -o -q "/content/dataset.zip" -d /content/

In [ ]:
!find "/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui" -maxdepth 3 

/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui
/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui/Epic and CSCR hospital Dataset.zip


In [11]:
!unzip  -o -q "/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui/Epic and CSCR hospital Dataset.zip" -d /content/
!find "/content/" -maxdepth 3 

/content/
/content/.config
/content/.config/default_configs.db
/content/.config/.last_opt_in_prompt.yaml
/content/.config/active_config
/content/.config/.last_survey_prompt.yaml
/content/.config/gce
/content/.config/.last_update_check.json
/content/.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
/content/.config/logs
/content/.config/logs/2026.03.30
/content/.config/configurations
/content/.config/configurations/config_default
/content/.config/config_sentinel
/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui
/content/Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui/Epic and CSCR hospital Dataset.zip
/content/Epic and CSCR hospital Dataset
/content/Epic and CSCR hospital Dataset/Train
/content/Epic and CSCR hospital Dataset/Train/glioma
/content/Epic and CSCR hospital Dataset/Train/notumor
/content/Epic and CSCR hospital Dataset/Train/meningioma
/content/Epic and CSCR hospital Dataset/Train/pituitary
/content/Epic and CSCR hospital Dataset/Test
/conten

In [12]:
!mv "/content/Epic and CSCR hospital Dataset/Train" "/content/Epic and CSCR hospital Dataset/train" 
!mv "Epic and CSCR hospital Dataset/train" "/content/" 

In [17]:
!mkdir "/content/target/"
!mv "/content/train" "/content/target/train" 
!mv "/content/train" "/content/target/train" 

mv: cannot stat '/content/train': No such file or directory


In [14]:
!mv "/content/Epic and CSCR hospital Dataset/Test" "/content/Epic and CSCR hospital Dataset/test" 
!mv "Epic and CSCR hospital Dataset/test" "/content/" 

In [ ]:
!mv "/content/test" "/content/target/test" 


mv: cannot stat '/content/test': No such file or directory


In [61]:

#from google.colab import drive
#drive.mount("/content/drive")
#DRIVE_PATH = Path("/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/")
DRIVE_PATH = Path("/content/")

In [30]:
!wget "https://data.mendeley.com/public-api/zip/w4sw3s9f59/download/1" -O external-validation.zip



--2026-04-08 13:19:56--  https://data.mendeley.com/public-api/zip/w4sw3s9f59/download/1
Resolving data.mendeley.com (data.mendeley.com)... 162.159.133.86, 162.159.130.86
Connecting to data.mendeley.com (data.mendeley.com)|162.159.133.86|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://prod-dcd-datasets-cache-zipfiles.s3.eu-west-1.amazonaws.com/w4sw3s9f59-1.zip?X-Amz-Security-Token=IQoJb3JpZ2luX2VjEDUaCWV1LXdlc3QtMSJGMEQCIEevIybZ5UN5g4hhsiJ1vhxdRQ2tEGAQgVs5tBSHg2ePAiADfmb3gXmO%2FZUAe8lctUQOLLBEhcgldtAI72J4bTBMgSqVBQj%2B%2F%2F%2F%2F%2F%2F%2F%2F%2F%2F8BEAQaDDM2NzE0NzM4MzgyNSIMl31utVHlVzAFjfiWKukEGP6zndztamfXYb87rtqlM8fy%2BPpKzhq8iWENT2sTUrFmyVtTBankRR6e5awYSIfv19LnRZmz7FpvBqVp4jlvnbyPivHuqSxZwBUWVaS98fAogzk8ATGaiJt9hXMdADyxOrynhDu0cvoBFLljxBWoPcyS8jnPYBk1BERZAFkvkiPSabT9ru4Y%2Ft8dYEHgq%2Bd3D825kZP4eENngyg4%2FSvoABD69rY4cNYx50z%2FwWHgvukBoIErNjptcTQ%2FRYxuGu7%2BJzjuWKFvXhtyhsE6ZvAJmraL0kVBfQGpGU2oRWfOnkocoVbArkCVJBLxuuJZh1jLOYFji8trg%2FAYZhSqpswewHCZ%2

In [ ]:
!unzip  -o -q "/content/external-validation.zip" -d /content/


In [33]:
!unzip  -o -q "/content/Brain Tumor Data/Brain Tumor data.zip" -d /content/
!ls "/content/"

'Brain Tumor data'				       external-validation.zip
'Brain Tumor Data'				       sample_data
'Brain Tumor MRI Dataset (Glioma, Meningioma, Pitui'   source
 dataset.zip					       target
'Epic and CSCR hospital Dataset'


In [35]:
# 1. Create the destination folder
!mkdir -p /content/external-validation-1/

# 2. Rename folders if they exist in the old location
![ -d "/content/Brain Tumor data/Training" ] && mv "/content/Brain Tumor data/Training" "/content/Brain Tumor data/train"
![ -d "/content/Brain Tumor data/Testing" ] && mv "/content/Brain Tumor data/Testing" "/content/Brain Tumor data/test"

# 3. Move them to the final destination (using -f to force)
!mv -f "/content/Brain Tumor data/train" "/content/external-validation-1/" 2>/dev/null || echo "Train already moved"
!mv -f "/content/Brain Tumor data/test" "/content/external-validation-1/" 2>/dev/null || echo "Test already moved"

# 4. Final verification
!ls -F /content/external-validation-1/

Train already moved
Test already moved
test/  train/


In [42]:
import kagglehub

# Download latest version
external_validation_2_path = kagglehub.dataset_download("alamshihab075/brain-tumor-mri-dataset-for-deep-learning")


print("Path to dataset files:", external_validation_2_path)

Using Colab cache for faster access to the 'brain-tumor-mri-dataset-for-deep-learning' dataset.
Path to dataset files: /kaggle/input/brain-tumor-mri-dataset-for-deep-learning


In [43]:
!find "{external_validation_2_path}" -maxdepth 3 -type d

/kaggle/input/brain-tumor-mri-dataset-for-deep-learning
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test/Pituitary
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test/No Tumor
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test/Meningioma
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test/Glioma
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train/Pituitary
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train/No Tumor
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train/Meningioma
/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train/Glioma


In [47]:
# 1. Create the destination directory
!mkdir -p /content/external-validation-2/

# 2. Find the deep 'Train' folder, copy it to destination as 'train'
# We use cp -r because kaggle/input is read-only
!mv -r "/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/Train/Train" /content/external-validation-2/train

# 3. Find the deep 'test' folder, copy it to destination as 'test'
!mv -r "/kaggle/input/brain-tumor-mri-dataset-for-deep-learning/test/test" "/content/external-validation-2/test"

# 4. Verify the move
!ls -R /content/external-validation-2/

/content/external-validation-2/:
test  train

/content/external-validation-2/test:
 Glioma   Meningioma  'No Tumor'   Pituitary

/content/external-validation-2/test/Glioma:
images

/content/external-validation-2/test/Glioma/images:
'gg (10).jpg'	'gg (324).jpg'	'gg (532).jpg'	  Tr-gl_0495.jpg
'gg (131).jpg'	'gg (33).jpg'	'gg (536).jpg'	  Tr-gl_0505.jpg
'gg (132).jpg'	'gg (342).jpg'	'gg (538).jpg'	  Tr-gl_0507.jpg
'gg (135).jpg'	'gg (350).jpg'	'gg (54).jpg'	  Tr-gl_0508.jpg
'gg (136).jpg'	'gg (352).jpg'	'gg (550).jpg'	  Tr-gl_0509.jpg
'gg (140).jpg'	'gg (359).jpg'	'gg (553).jpg'	  Tr-gl_0518.jpg
'gg (141).jpg'	'gg (35).jpg'	'gg (558).jpg'	  Tr-gl_0541.jpg
'gg (21).jpg'	'gg (362).jpg'	'gg (568).jpg'	  Tr-gl_0549.jpg
'gg (221).jpg'	'gg (368).jpg'	'gg (57).jpg'	  Tr-gl_0551.jpg
'gg (227).jpg'	'gg (372).jpg'	'gg (59).jpg'	  Tr-gl_0640.jpg
'gg (229).jpg'	'gg (37).jpg'	'gg (66).jpg'	  Tr-gl_0675.jpg
'gg (22).jpg'	'gg (385).jpg'	'gg (72).jpg'	  Tr-gl_0691.jpg
'gg (239).jpg'	'gg (397).jpg'	'gg (

In [49]:
import kagglehub

# Download latest version
external_validation_3_path = kagglehub.dataset_download("deeppythonist/brain-tumor-mri-dataset")

print("Path to dataset files:", {external_validation_3_path})

Using Colab cache for faster access to the 'brain-tumor-mri-dataset' dataset.
Path to dataset files: {'/kaggle/input/brain-tumor-mri-dataset'}


In [50]:
!find "{external_validation_3_path}" -maxdepth 3 -type d

/kaggle/input/brain-tumor-mri-dataset
/kaggle/input/brain-tumor-mri-dataset/test
/kaggle/input/brain-tumor-mri-dataset/test/pituitary
/kaggle/input/brain-tumor-mri-dataset/test/notumor
/kaggle/input/brain-tumor-mri-dataset/test/meningioma
/kaggle/input/brain-tumor-mri-dataset/test/glioma
/kaggle/input/brain-tumor-mri-dataset/train
/kaggle/input/brain-tumor-mri-dataset/train/pituitary
/kaggle/input/brain-tumor-mri-dataset/train/notumor
/kaggle/input/brain-tumor-mri-dataset/train/meningioma
/kaggle/input/brain-tumor-mri-dataset/train/glioma


In [51]:
# 1. Set the path variable (assuming kagglehub download or direct kaggle input)
# Based on your input string, the base path is:
external_validation_3_path = "/kaggle/input/brain-tumor-mri-dataset"

# 2. Create the destination directory in /content/
!mkdir -p /content/external-validation-3/

# 3. Copy the folders. 
# We use the path you provided to target the correct subdirectories.
!cp -r "{external_validation_3_path}/train" /content/external-validation-3/train
!cp -r "{external_validation_3_path}/test" /content/external-validation-3/test

# 4. Verify the move and structure
print("Contents of external-validation-3:")
!ls -F /content/external-validation-3/

Contents of external-validation-3:
test/  train/


In [52]:
# 1. Create the new parent directories
!mkdir -p /content/DANN/
!mkdir -p "/content/External Validation/"

# 2. Move source and target into /content/DANN/
# (Checks if they exist first to avoid errors)
![ -d "/content/source" ] && mv /content/source /content/DANN/
![ -d "/content/target" ] && mv /content/target /content/DANN/

# 3. Move all external-validation folders into the new "External Validation" dir
# Using a wildcard * to grab 1, 2, and 3 at once
!mv /content/external-validation-* "/content/External Validation/"

# 4. Final verification
!echo "--- DANN Folder ---"
!ls -F /content/DANN/
!echo -e "\n--- External Validation Folder ---"
!ls -F "/content/External Validation/"

--- DANN Folder ---
source/  target/

--- External Validation Folder ---
external-validation-1/	external-validation-2/	external-validation-3/


In [54]:
!mv "/content/External Validation/external-validation-1" "/content/External Validation/External-validation-dataset-1"
!mv "/content/External Validation/external-validation-2" "/content/External Validation/External-validation-dataset-2"
!mv "/content/External Validation/external-validation-3" "/content/External Validation/External-validation-dataset-3"

# 3. Final Check
!ls "/content/External Validation/"

mv: cannot stat '/content/External Validation/external-validation-1': No such file or directory
mv: cannot stat '/content/External Validation/external-validation-2': No such file or directory
mv: cannot stat '/content/External Validation/external-validation-3': No such file or directory
External-validation-dataset-1  External-validation-dataset-3
External-validation-dataset-2


In [ ]:
!find "{DRIVE_PATH}" -maxdepth 3 -type d

/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1/test
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-validation-dataset-1/train
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2/test
/content/drive/MyDrive/Mini Project Sem-6(Brain Tumor Classification)/MRI-Dataset/External Validation/External-vaildation-dataset-2/train
/

In [62]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

# =====================================================================
# 1. SETUP & CONFIGURATION
# =====================================================================

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif", ".webp"}
MAX_WORKERS = 8

# Grouped dictionary: Priority 1 = Keep, Priority 5 = Delete.
DATASETS = {
    "Source_BRISC":    {"path": DRIVE_PATH / "DANN/source", "priority": 1},
    "Target_Mendeley": {"path": DRIVE_PATH / "DANN/target", "priority": 2},
    "Val_Ayesha":      {"path": DRIVE_PATH / "External Validation/External-validation-dataset-1", "priority": 3},
    "Val_Alam":        {"path": DRIVE_PATH / "External Validation/External-validation-dataset-2", "priority": 4},
    "Val_DeepPy":      {"path": DRIVE_PATH / "External Validation/External-validation-dataset-3", "priority": 5},
}

# Sort datasets by priority for consistent output
DATASETS_BY_PRIORITY = sorted(DATASETS.items(), key=lambda x: x[1]["priority"])


# =====================================================================
# 2. PATH VALIDATION & PRIORITY CHECK
# =====================================================================
print("Priority order (1=keep, 5=delete):")
for name, cfg in DATASETS_BY_PRIORITY:
    exists = cfg["path"].exists()
    status = "✅" if exists else "❌ PATH NOT FOUND"
    print(f"  [{cfg['priority']}] {name:<20} → {cfg['path']}  {status}")


# =====================================================================
# 3. UTILITIES
# =====================================================================
def count_images(folder_path):
    """Recursively counts images in the given directory using the global VALID_EXTENSIONS."""
    directory = Path(folder_path)
    
    if not directory.exists():
        return "Folder not found"
        
    return sum(
        1 for file in directory.rglob('*') 
        if file.is_file() and file.suffix.lower() in VALID_EXTENSIONS
    )


# =====================================================================
# 4. EXECUTION & REPORTING
# =====================================================================
print(f"\n{'Dataset Name':<20} | {'Train Images':<15} | {'Test Images'}")
print("-" * 60)

# Iterate through the sorted list to keep the output in order of priority
for name, cfg in DATASETS_BY_PRIORITY:
    base_path = cfg["path"]
    
    # Path library makes joining folders as simple as using the '/' operator
    train_count = count_images(base_path / "train")
    test_count = count_images(base_path / "test")
    
    print(f"{name:<20} | {str(train_count):<15} | {str(test_count)}")

Priority order (1=keep, 5=delete):
  [1] Source_BRISC         → /content/DANN/source  ✅
  [2] Target_Mendeley      → /content/DANN/target  ✅
  [3] Val_Ayesha           → /content/External Validation/External-validation-dataset-1  ✅
  [4] Val_Alam             → /content/External Validation/External-validation-dataset-2  ✅
  [5] Val_DeepPy           → /content/External Validation/External-validation-dataset-3  ✅

Dataset Name         | Train Images    | Test Images
------------------------------------------------------------
Source_BRISC         | 5000            | 1000
Target_Mendeley      | 9650            | 2414
Val_Ayesha           | 5712            | 1311
Val_Alam             | 8745            | 512
Val_DeepPy           | 5723            | 1430


In [63]:
def collect_images(dataset_name: str, root_path: str):
    root = Path(root_path)
    if not root.exists():
        return []
    return [(dataset_name, str(p.resolve())) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS]

all_images = []
dataset_image_counts = {}

for name, cfg in DATASETS_BY_PRIORITY:
    imgs = collect_images(name, cfg["path"])
    dataset_image_counts[name] = len(imgs)
    all_images.extend(imgs)
    print(f"  {name:<20}: {len(imgs):>5} images found")

print(f"\n📁 Total images to hash: {len(all_images):,}")

  Source_BRISC        :  6000 images found
  Target_Mendeley     : 12064 images found
  Val_Ayesha          :  7023 images found
  Val_Alam            :  9257 images found
  Val_DeepPy          :  7153 images found

📁 Total images to hash: 41,497


In [64]:
def compute_phash(filepath):
    try:
        with Image.open(filepath) as img:
            return str(imagehash.phash(img)) # Convert to string for stable matching
    except Exception:
        return None

hash_map = defaultdict(list)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(compute_phash, img[1]): img for img in all_images}
    
    for future in tqdm(as_completed(futures), total=len(all_images), desc="pHashing"):
        dataset_name, filepath = futures[future]
        h = future.result()
        if h is not None:
            hash_map[h].append((dataset_name, filepath))

print(f"✅ Hashing complete. Unique visual signatures found: {len(hash_map)}")

pHashing:   0%|          | 0/41497 [00:00<?, ?it/s]

✅ Hashing complete. Unique visual signatures found: 14412


In [65]:
leakage_matrix = defaultdict(lambda: defaultdict(int))
leakage_pairs = []       # Cross-dataset duplicates
internal_pairs = []      # Same-folder duplicates
internal_totals = defaultdict(int)

for hash_str, entries in hash_map.items():
    # --- 1. HANDLE INTERNAL DUPLICATES FIRST ---
    # Group entries by dataset to find duplicates sitting in the same folder
    ds_to_paths = defaultdict(list)
    for ds, path in entries:
        ds_to_paths[ds].append(path)
        
    unique_entries = [] 
    
    for ds, paths in ds_to_paths.items():
        # Keep the very first path we find for this specific dataset
        unique_entries.append((ds, paths[0]))
        
        # Any extra paths in the exact same dataset are internal duplicates
        for dup_path in paths[1:]:
            internal_pairs.append((ds, dup_path))
            internal_totals[ds] += 1
            
    # --- 2. HANDLE CROSS-DATASET LEAKAGE ---
    # If this hash only exists in one dataset (after internal cleanup), skip
    if len(unique_entries) < 2:
        continue 

    # Sort so the lowest priority number (highest importance) is first
    sorted_entries = sorted(unique_entries, key=lambda e: DATASETS[e[0]]["priority"])
    keeper_ds, keeper_path = sorted_entries[0]

    seen_datasets = {keeper_ds}
    for dup_ds, dup_path in sorted_entries[1:]:
        if dup_ds not in seen_datasets:
            leakage_pairs.append((keeper_ds, keeper_path, dup_ds, dup_path))
            leakage_matrix[keeper_ds][dup_ds] += 1
            seen_datasets.add(dup_ds)

print(f"🗑️  Internal duplicates found: {len(internal_pairs)}")
print(f"🔎 Cross-dataset duplicates found: {len(leakage_pairs)}")

🗑️  Internal duplicates found: 7947
🔎 Cross-dataset duplicates found: 19138


In [66]:
actual_deleted_internal = 0
actual_deleted_cross = 0

# 1. Delete Internal Duplicates
if internal_pairs:
    print(f"\n🧹 Starting deletion of {len(internal_pairs)} INTERNAL files...")
    for ds, del_path in tqdm(internal_pairs, desc="Internal Dels"):
        try:
            if os.path.exists(del_path):
                os.remove(del_path)
                actual_deleted_internal += 1
        except Exception as e:
            pass

# 2. Delete Cross-Dataset Duplicates
if leakage_pairs:
    print(f"\n⚠️  Starting deletion of {len(leakage_pairs)} CROSS-DATASET files...")
    for _, _, _, del_path in tqdm(leakage_pairs, desc="External Dels"):
        try:
            if os.path.exists(del_path):
                os.remove(del_path)
                actual_deleted_cross += 1
        except Exception as e:
            pass

print(f"\n✅ Successfully removed {actual_deleted_internal} internal copies.")
print(f"✅ Successfully removed {actual_deleted_cross} cross-dataset leaks.")


🧹 Starting deletion of 7947 INTERNAL files...


Internal Dels:   0%|          | 0/7947 [00:00<?, ?it/s]


⚠️  Starting deletion of 19138 CROSS-DATASET files...


External Dels:   0%|          | 0/19138 [00:00<?, ?it/s]


✅ Successfully removed 7947 internal copies.
✅ Successfully removed 19138 cross-dataset leaks.


In [69]:
COL_W, LABEL_W = 18, 20 
dataset_names = [name for name, _ in DATASETS_BY_PRIORITY]
print("\n" + "═" * (LABEL_W + COL_W * len(dataset_names) + 4))
print(" 🔬 CROSS-DATASET LEAKAGE MATRIX")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 4))
print("\n  Rows = SOURCE (kept)   │   Cols = TARGET (deleted from)\n")

header = f"{'SOURCE \\ TARGET':<{LABEL_W}}"
for col in dataset_names:
    p = DATASETS[col]["priority"]
    header += f"[P{p}]{col:<{COL_W - 4}}"
print(header)
print("─" * len(header))

col_totals = defaultdict(int)
for row_name in dataset_names:
    p = DATASETS[row_name]["priority"]
    row_str = f"[P{p}]{row_name:<{LABEL_W - 4}}"
    row_sum = 0
    for col_name in dataset_names:
        if col_name == row_name:
            row_str += f"{'—':>{COL_W}}"
        else:
            count = leakage_matrix[row_name].get(col_name, 0)
            row_str += f"{str(count) if count > 0 else '0':>{COL_W}}"
            row_sum += count
            col_totals[col_name] += count
    print(row_str + f"  │ Total external deleted: {row_sum}")

print("─" * len(header))
totals_row = f"{'Col Total (deleted)':<{LABEL_W}}"
for col_name in dataset_names:
    totals_row += f"{col_totals.get(col_name, 0):>{COL_W}}"
print(totals_row)

print("\n" + "═" * (LABEL_W + COL_W * len(dataset_names) + 40))
print(" 📋 GLOBAL DEDUPLICATION SUMMARY")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 40))

for name in dataset_names:
    p = DATASETS[name]["priority"]
    original_count = dataset_image_counts.get(name, 0)
    
    internal_deleted = internal_totals.get(name, 0)
    external_deleted = col_totals.get(name, 0)   
    remaining = original_count - internal_deleted - external_deleted
    
    print(f"  [P{p}] {name:<20} | Original: {original_count:>5} | "
          f"Internal Dels: {internal_deleted:>5} | "
          f"External Dels: {external_deleted:>5} | "
          f"Remaining: {remaining:>5}")

print(f"\n  {'Total Internal Copies Removed':.<40} {actual_deleted_internal}")
print(f"  {'Total Cross-Dataset Leaks Removed':.<40} {actual_deleted_cross}")
print(f"  {'GRAND TOTAL DELETED':.<40} {actual_deleted_internal + actual_deleted_cross}")
print("═" * (LABEL_W + COL_W * len(dataset_names) + 40))


══════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 🔬 CROSS-DATASET LEAKAGE MATRIX
══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Rows = SOURCE (kept)   │   Cols = TARGET (deleted from)

SOURCE \ TARGET     [P1]Source_BRISC  [P2]Target_Mendeley[P3]Val_Ayesha    [P4]Val_Alam      [P5]Val_DeepPy    
───────────────────────────────────────────────────────────────────────────────────────────────────────────────
[P1]Source_BRISC                     —              4057              4689              2765              4689  │ Total external deleted: 16200
[P2]Target_Mendeley                  0                 —               911               745               911  │ Total external deleted: 2567
[P3]Val_Ayesha                       0                 0                 —                61               310  │ Total external deleted: 371
[P4]Val_Alam            